# 🔐 Network Intrusion Detection System — Upgraded (4 ML Layers)

**Architecture:**
- 🔵 Layer 1 — Base Models: Logistic Regression, Random Forest, XGBoost
- 🟢 Layer 2 — Improved Deep Neural Network (BatchNorm + LeakyReLU)
- 🟡 Layer 3 — Stacking Ensemble (meta-learner on top of all models)
- 🔴 Layer 4 — Autoencoder for Anomaly Detection (catches unknown attacks)

> ✅ Runs 100% inside Google Colab — no local setup required.

---

## ✅ Step 1 — Install Libraries

In [ ]:
!pip install xgboost imbalanced-learn scikit-learn seaborn matplotlib pandas numpy tensorflow -q
print('✅ All libraries installed!')

## ✅ Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib

# Check GPU
print('✅ Libraries imported!')
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## ✅ Step 3 — Download Dataset

In [ ]:
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt -O KDDTrain+.txt
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt  -O KDDTest+.txt
print('✅ Dataset downloaded!')

## ✅ Step 4 — Load & Preprocess Data

In [ ]:
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  names=columns)

train_df.drop('difficulty', axis=1, inplace=True)
test_df.drop('difficulty',  axis=1, inplace=True)

# Binary labels
train_df['binary_label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['binary_label']  = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)

# Encode categoricals
combined = pd.concat([train_df, test_df])
le = LabelEncoder()
for col in ['protocol_type', 'service', 'flag']:
    combined[col] = le.fit_transform(combined[col])
train_df = combined[:len(train_df)].copy()
test_df  = combined[len(train_df):].copy()

drop_cols = ['label', 'binary_label']
X_train = train_df.drop(drop_cols, axis=1)
y_train = train_df['binary_label']
X_test  = test_df.drop(drop_cols, axis=1)
y_test  = test_df['binary_label']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

print(f'✅ Preprocessing done!')
print(f'Train: {X_res.shape}, Test: {X_test_scaled.shape}')
print(f'Class balance after SMOTE: {pd.Series(y_res).value_counts().to_dict()}')

---
# 🔵 LAYER 1 — Base Models

In [ ]:
print('🔵 Training Layer 1 — Base Models...')

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_res, y_res)
lr_preds = lr.predict(X_test_scaled)
print(f'  Logistic Regression Accuracy : {accuracy_score(y_test, lr_preds):.4f}')

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_res, y_res)
rf_preds = rf.predict(X_test_scaled)
print(f'  Random Forest Accuracy       : {accuracy_score(y_test, rf_preds):.4f}')

# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                    random_state=42, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_res, y_res)
xgb_preds = xgb.predict(X_test_scaled)
print(f'  XGBoost Accuracy             : {accuracy_score(y_test, xgb_preds):.4f}')

print('✅ Layer 1 complete!')

---
# 🟢 LAYER 2 — Improved Deep Neural Network

In [ ]:
print('🟢 Building Layer 2 — Deep Neural Network...')

def build_deep_nn(input_dim):
    inputs = keras.Input(shape=(input_dim,))

    # Block 1
    x = layers.Dense(256)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.3)(x)

    # Block 2
    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Dense(64)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.2)(x)

    # Block 4
    x = layers.Dense(32)(x)
    x = layers.LeakyReLU(alpha=0.1)(x)

    # Output
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

nn_model = build_deep_nn(X_res.shape[1])
nn_model.summary()

In [ ]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

history = nn_model.fit(
    X_res, y_res,
    epochs=50,
    batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

nn_preds = (nn_model.predict(X_test_scaled) > 0.5).astype(int).flatten()
print(f'\n✅ Deep Neural Network Accuracy: {accuracy_score(y_test, nn_preds):.4f}')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Deep NN — Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Deep NN — Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

---
# 🟡 LAYER 3 — Stacking Ensemble

In [ ]:
print('🟡 Building Layer 3 — Stacking Ensemble...')

# Get probability predictions from all base models
lr_prob  = lr.predict_proba(X_test_scaled)[:, 1]
rf_prob  = rf.predict_proba(X_test_scaled)[:, 1]
xgb_prob = xgb.predict_proba(X_test_scaled)[:, 1]
nn_prob  = nn_model.predict(X_test_scaled).flatten()

# Also get train probabilities for meta-learner training
lr_prob_train  = lr.predict_proba(X_res)[:, 1]
rf_prob_train  = rf.predict_proba(X_res)[:, 1]
xgb_prob_train = xgb.predict_proba(X_res)[:, 1]
nn_prob_train  = nn_model.predict(X_res).flatten()

# Stack predictions as new features
meta_X_train = np.column_stack([lr_prob_train, rf_prob_train, xgb_prob_train, nn_prob_train])
meta_X_test  = np.column_stack([lr_prob, rf_prob, xgb_prob, nn_prob])

# Meta-learner: Logistic Regression on top
meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner.fit(meta_X_train, y_res)
stack_preds = meta_learner.predict(meta_X_test)

print(f'✅ Stacking Ensemble Accuracy: {accuracy_score(y_test, stack_preds):.4f}')
print(f'   Stacking Ensemble ROC-AUC: {roc_auc_score(y_test, stack_preds):.4f}')

---
# 🔴 LAYER 4 — Autoencoder for Anomaly Detection

In [ ]:
print('🔴 Building Layer 4 — Autoencoder...')

# Train ONLY on normal traffic
X_normal = X_train_scaled[y_train == 0]
print(f'Normal traffic samples for autoencoder training: {X_normal.shape[0]}')

def build_autoencoder(input_dim):
    # Encoder
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='relu')(inputs)
    x = layers.Dense(16, activation='relu')(x)
    encoded = layers.Dense(8, activation='relu')(x)

    # Decoder
    x = layers.Dense(16, activation='relu')(encoded)
    x = layers.Dense(32, activation='relu')(x)
    decoded = layers.Dense(input_dim, activation='linear')(x)

    autoencoder = keras.Model(inputs, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

autoencoder = build_autoencoder(X_normal.shape[1])
autoencoder.summary()

In [ ]:
ae_early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

ae_history = autoencoder.fit(
    X_normal, X_normal,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[ae_early_stop],
    verbose=1
)
print('✅ Autoencoder trained!')

In [ ]:
# Compute reconstruction error on test data
reconstructed = autoencoder.predict(X_test_scaled)
mse_errors = np.mean(np.power(X_test_scaled - reconstructed, 2), axis=1)

# Find threshold: 95th percentile of normal traffic reconstruction error
normal_recon = autoencoder.predict(X_normal)
normal_errors = np.mean(np.power(X_normal - normal_recon, 2), axis=1)
threshold = np.percentile(normal_errors, 95)
print(f'Anomaly threshold (95th percentile): {threshold:.6f}')

# Predict: high reconstruction error = Attack
ae_preds = (mse_errors > threshold).astype(int)
print(f'✅ Autoencoder Accuracy : {accuracy_score(y_test, ae_preds):.4f}')
print(f'   Autoencoder ROC-AUC : {roc_auc_score(y_test, ae_preds):.4f}')

In [ ]:
# Visualize reconstruction error distribution
plt.figure(figsize=(10, 5))
plt.hist(mse_errors[y_test == 0], bins=100, alpha=0.6, label='Normal Traffic', color='green')
plt.hist(mse_errors[y_test == 1], bins=100, alpha=0.6, label='Attack Traffic', color='red')
plt.axvline(threshold, color='black', linestyle='--', label=f'Threshold = {threshold:.4f}')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.title('Autoencoder — Reconstruction Error Distribution')
plt.legend()
plt.tight_layout()
plt.show()

---
# 📊 Final Evaluation — All 4 Layers

In [ ]:
all_models = {
    '🔵 Logistic Regression (L1)' : lr_preds,
    '🔵 Random Forest (L1)'       : rf_preds,
    '🔵 XGBoost (L1)'             : xgb_preds,
    '🟢 Deep Neural Network (L2)' : nn_preds,
    '🟡 Stacking Ensemble (L3)'   : stack_preds,
    '🔴 Autoencoder (L4)'         : ae_preds,
}

results = []
for name, preds in all_models.items():
    results.append({
        'Model'   : name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'ROC-AUC' : round(roc_auc_score(y_test, preds), 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['Accuracy'], width, label='Accuracy',  color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, results_df['ROC-AUC'],  width, label='ROC-AUC',   color='coral',     edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=9)
ax.set_ylim(0.75, 1.0)
ax.set_title('All 4 Layers — Accuracy & ROC-AUC Comparison')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, preds) in enumerate(all_models.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    axes[idx].set_title(name, fontsize=9)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All 4 Layers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(9, 6))

roc_probs = {
    '🔵 Logistic Regression (L1)' : lr.predict_proba(X_test_scaled)[:, 1],
    '🔵 Random Forest (L1)'       : rf.predict_proba(X_test_scaled)[:, 1],
    '🔵 XGBoost (L1)'             : xgb.predict_proba(X_test_scaled)[:, 1],
    '🟢 Deep Neural Network (L2)' : nn_model.predict(X_test_scaled).flatten(),
    '🟡 Stacking Ensemble (L3)'   : meta_learner.predict_proba(meta_X_test)[:, 1],
    '🔴 Autoencoder (L4)'         : mse_errors,
}

for name, prob in roc_probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')

plt.plot([0,1],[0,1],'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — All 4 Layers')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed report for best model (Stacking Ensemble)
print('=== 🟡 Stacking Ensemble — Detailed Classification Report ===')
print(classification_report(y_test, stack_preds, target_names=['Normal', 'Attack']))

## ✅ Step — Feature Importance (Random Forest)

In [ ]:
feat_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf.feature_importances_})
feat_df = feat_df.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_df, palette='viridis')
plt.title('Top 15 Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

## ✅ Save All Models (Download to PC)

In [ ]:
# Save all models inside Colab
joblib.dump(lr,           'model_lr.pkl')
joblib.dump(rf,           'model_rf.pkl')
joblib.dump(xgb,          'model_xgb.pkl')
joblib.dump(meta_learner, 'model_meta_learner.pkl')
joblib.dump(scaler,       'scaler.pkl')
nn_model.save('model_deep_nn.h5')
autoencoder.save('model_autoencoder.h5')
print('✅ All models saved inside Colab!')

# Download to your PC
from google.colab import files
for fname in ['model_rf.pkl', 'model_xgb.pkl', 'model_meta_learner.pkl',
              'model_deep_nn.h5', 'model_autoencoder.h5', 'scaler.pkl']:
    files.download(fname)
print('✅ Downloads started!')

## ✅ Quick Prediction Demo

In [ ]:
label_map = {0: 'Normal ✅', 1: 'Attack 🚨'}
samples = X_test_scaled[:5]
actual  = y_test.values[:5]

# Stacking ensemble prediction pipeline
s_lr  = lr.predict_proba(samples)[:, 1]
s_rf  = rf.predict_proba(samples)[:, 1]
s_xgb = xgb.predict_proba(samples)[:, 1]
s_nn  = nn_model.predict(samples).flatten()
s_meta = np.column_stack([s_lr, s_rf, s_xgb, s_nn])
final_preds = meta_learner.predict(s_meta)

print('Final Predictions via Stacking Ensemble:')
print('-' * 45)
for i in range(5):
    print(f'Sample {i+1} | Actual: {label_map[actual[i]]:15s} | Predicted: {label_map[final_preds[i]]}')

---
## 🎉 Project Complete — 4 ML Layers!

| Layer | Model | Purpose |
|-------|-------|---------|
| 🔵 Layer 1 | LR + RF + XGBoost | Strong base classifiers |
| 🟢 Layer 2 | Deep Neural Network | BatchNorm + LeakyReLU + EarlyStopping |
| 🟡 Layer 3 | Stacking Ensemble | Combines all model predictions |
| 🔴 Layer 4 | Autoencoder | Detects unknown/zero-day attacks |

> ✅ Entire project ran 100% inside Google Colab — no local setup needed!